The original purpose of this code is to combine the categorized queries and the ndcg@10 values ​​from all scores into a large table.

I have already completed this task in nf_cats_all_datasets.py, combining the scores and classification results of all datasets to generate nf_cats_all_datasets_with_scores.csv.

The current task is to perform statistical analysis on the combined scores.

I want to perform LMM test analysis on the results of each model at the query granularity obtained from the experiment. This code also generates the file data/df_long_all_model_scores.csv used for LMM analysis.


In [ ]:
# 1， Load data
import numpy as np
import pandas as pd

data = pd.read_csv('data/nf_cats_all_datasets_with_scores.csv')
data

,query_id,dataset_name,raw_query_id,query_type,query_text,split,bm25_score_ndcg@10,contriever_score_ndcg@10,bge_m3_score_ndcg@10,qwen3_score_ndcg@10,linq_score_ndcg@10,gte_score_ndcg@10,reasonir_score_ndcg@10,diver_score_ndcg@10,bge_reasoner_score_ndcg@10
0,trec-covid__test__1,trec-covid,1,EVIDENCE-BASED,what is the origin of COVID-19,test,0.252841,0.039199,0.340113,1.000000,0.716937,0.784617,0.653408,0.501819,0.778429
1,trec-covid__test__2,trec-covid,2,EVIDENCE-BASED,how does the coronavirus respond to changes in...,test,0.613283,0.252841,0.864315,1.000000,1.000000,0.861138,0.926636,1.000000,0.921602
2,trec-covid__test__3,trec-covid,3,DEBATE,will SARS-CoV2 infected people develop immunit...,test,0.195189,0.894275,0.000000,1.000000,0.930569,0.899030,0.827749,0.826422,0.650414
3,trec-covid__test__4,trec-covid,4,EVIDENCE-BASED,what causes death from Covid-19?,test,0.135685,0.110046,0.314880,1.000000,0.966873,0.965284,0.591945,0.957428,0.831848
4,trec-covid__test__5,trec-covid,5,EVIDENCE-BASED,what drugs have been active against SARS-CoV o...,test,0.519520,0.508287,0.283713,1.000000,0.743345,0.546287,0.706663,0.671759,0.571936
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42519,msmarco__test__1132532,msmarco,1132532,FACTOID,average annual income data analyst,test,0.428797,0.835746,0.889954,1.000000,0.978793,0.976856,0.776182,0.770499,0.898255
42520,msmarco__test__1133579,msmarco,1133579,EVIDENCE-BASED,how does granulation tissue start,test,0.554417,0.629336,0.812214,0.818852,0.817511,0.627877,0.652478,0.676379,0.741429
42521,msmarco__test__1136043,msmarco,1136043,FACTOID,difference between a hotel and motel,test,0.346317,0.805447,0.724774,0.725397,0.724774,0.723278,0.618023,0.683002,0.593403
42522,msmarco__test__1136047,msmarco,1136047,FACTOID,difference between a company's strategy and bu...,test,0.459095,0.327582,0.475790,0.557774,0.613942,0.371733,0.433620,0.461970,0.305609


In [ ]:
# 2, Statistical Analysis

# Statistically analyze the performance of each model on each specific dataset (dataset_name), calculating the mean and standard deviation of ndcg@10.

# This yields the performance on each dataset, which should be consistent with the wandb results.
model_name_list = ['bm25','contriever', 'bge_m3', 'qwen3', 'linq', 'gte', 'reasonir', 'diver', 'bge_reasoner']

dataset_results = []
for model in model_name_list:
    col = f"{model}_score_ndcg@10"
    grouped_dataset = data.groupby(['dataset_name'])[col].agg(
        ['mean', 'std', 'min', 'max', 'count']).reset_index()
    print(f"Processed model: {model}")
    print(grouped_dataset)


Processed model: bm25
           dataset_name      mean       std       min       max  count
0                  aops  0.061984  0.149995  0.000000  0.765361    111
1               arguana  0.386585  0.352681  0.000000  1.000000   1406
2               biology  0.188905  0.273033  0.000000  1.000000    103
3       browsecomp_plus  0.043886  0.119771  0.000000  0.831872    830
4         climate-fever  0.151630  0.236297  0.000000  1.000000   1535
5        dbpedia-entity  0.309808  0.269749  0.000000  1.000000    400
6         earth_science  0.271736  0.298246  0.000000  1.000000    116
7             economics  0.148705  0.268324  0.000000  1.000000    103
8                 fever  0.602849  0.385046  0.000000  1.000000   6666
9                  fiqa  0.236742  0.319568  0.000000  1.000000    648
10             hotpotqa  0.600760  0.278795  0.000000  1.000000   7405
11             leetcode  0.241021  0.339225  0.000000  1.000000    142
12              msmarco  0.221301  0.326066  0.000000  

In [ ]:
# First, add the task type corresponding to the datasets.
task_types = {
    "msmarco":"Passage Retrieval",
    "trec-covid":"Bio-Medical Retrieval",
    "nfcorpus": "Bio-Medical Retrieval",
    "hotpotqa":"Question Answering",
    "fiqa":"Question Answering",
    "arguana":"Argument Retrieval",
    "quora":"Duplicate Question Retrieval",
    "dbpedia-entity":"Entity Retrieval",
    "scidocs": "Citation-Prediction",
    "nq": "Question Answering",
    "fever":"Fact Checking",
    "climate-fever": "Fact Checking",
    "scifact": "Fact Checking",
    "webis-touche2020":"Argument Retrieval",
    "biology":"StackExchange Post Retrieval",
    "earth_science":"StackExchange Post Retrieval",
    "economics":"StackExchange Post Retrieval",
    "psychology" :"StackExchange Post Retrieval",
    "robotics":"StackExchange Post Retrieval",
    "stackoverflow":"StackExchange Post Retrieval",
    "sustainable_living":"StackExchange Post Retrieval",
    "leetcode":'Code Retrieval',
    "pony":"Code Retrieval",
    "aops" :"Theorem Retrieval",
    "theoremqa_theorems" :"Theorem Retrieval",
    "theoremqa_questions" :"Theorem Retrieval",
    "browsecomp_plus":"Question Answering",
}
# Add a task type column
data['task_type'] = data['dataset_name'].map(task_types)
data

,query_id,dataset_name,raw_query_id,query_type,query_text,split,bm25_score_ndcg@10,contriever_score_ndcg@10,bge_m3_score_ndcg@10,qwen3_score_ndcg@10,linq_score_ndcg@10,gte_score_ndcg@10,reasonir_score_ndcg@10,diver_score_ndcg@10,bge_reasoner_score_ndcg@10,task_type
0,trec-covid__test__1,trec-covid,1,EVIDENCE-BASED,what is the origin of COVID-19,test,0.252841,0.039199,0.340113,1.000000,0.716937,0.784617,0.653408,0.501819,0.778429,Bio-Medical Retrieval
1,trec-covid__test__2,trec-covid,2,EVIDENCE-BASED,how does the coronavirus respond to changes in...,test,0.613283,0.252841,0.864315,1.000000,1.000000,0.861138,0.926636,1.000000,0.921602,Bio-Medical Retrieval
2,trec-covid__test__3,trec-covid,3,DEBATE,will SARS-CoV2 infected people develop immunit...,test,0.195189,0.894275,0.000000,1.000000,0.930569,0.899030,0.827749,0.826422,0.650414,Bio-Medical Retrieval
3,trec-covid__test__4,trec-covid,4,EVIDENCE-BASED,what causes death from Covid-19?,test,0.135685,0.110046,0.314880,1.000000,0.966873,0.965284,0.591945,0.957428,0.831848,Bio-Medical Retrieval
4,trec-covid__test__5,trec-covid,5,EVIDENCE-BASED,what drugs have been active against SARS-CoV o...,test,0.519520,0.508287,0.283713,1.000000,0.743345,0.546287,0.706663,0.671759,0.571936,Bio-Medical Retrieval
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42519,msmarco__test__1132532,msmarco,1132532,FACTOID,average annual income data analyst,test,0.428797,0.835746,0.889954,1.000000,0.978793,0.976856,0.776182,0.770499,0.898255,Passage Retrieval
42520,msmarco__test__1133579,msmarco,1133579,EVIDENCE-BASED,how does granulation tissue start,test,0.554417,0.629336,0.812214,0.818852,0.817511,0.627877,0.652478,0.676379,0.741429,Passage Retrieval
42521,msmarco__test__1136043,msmarco,1136043,FACTOID,difference between a hotel and motel,test,0.346317,0.805447,0.724774,0.725397,0.724774,0.723278,0.618023,0.683002,0.593403,Passage Retrieval
42522,msmarco__test__1136047,msmarco,1136047,FACTOID,difference between a company's strategy and bu...,test,0.459095,0.327582,0.475790,0.557774,0.613942,0.371733,0.433620,0.461970,0.305609,Passage Retrieval


In [ ]:
# 2, Statistical Analysis

# Perform statistical analysis on the performance of each model on each specific task, calculate the mean and standard deviation of ndcg@10, and calculate the average performance at the task level.
 

task_list = ['Passage Retrieval','Bio-Medical Retrieval','Question Answering','Duplicate Question Retrieval','Entity Retrieval','Citation-Prediction', 'Fact Checking', \
 'Argument Retrieval','StackExchange Post Retrieval', 'Code Retrieval', 'Theorem Retrieval', ]


for model in model_name_list:
    col = f"{model}_score_ndcg@10"
    grouped_task = data.groupby(['task_type'])[col].agg(
        ['mean', 'std', 'min', 'max', 'count']).reset_index()
    # Add model column to grouped_task
    grouped_task['model'] = model
    
    # Calculate the average task-level performance.
    overall_mean = grouped_task['mean'].mean()
    print(f"Processed model: {model}")
    # print(grouped_task)
    print(f"Task-Level Overall Mean for model {model}: {overall_mean}")

    # Add a row to calculate statistics for all queries.
    overall_stats = data[col].agg(['mean', 'std', 'min', 'max', 'count']).to_frame().T
    overall_stats['task_type'] = 'ALL-query-level'
    overall_stats['model'] = model
    grouped_task = pd.concat([grouped_task, overall_stats], ignore_index=True)

    # Reorder the rows according to the query_type_list, while retaining the newly added 'ALL' rows.
    grouped_task = grouped_task.set_index('task_type').reindex(task_list + ['ALL-query-level']).reset_index()

    print(f"Model: {model} - Task Type Performance:")
    print(grouped_task)
    print("-" * 50)

Processed model: bm25
Task-Level Overall Mean for model bm25: 0.325671651752591
Model: bm25 - Task Type Performance:
                       task_type      mean       std  min  max    count model
0              Passage Retrieval  0.221301  0.326066  0.0  1.0   7077.0  bm25
1          Bio-Medical Retrieval  0.342846  0.321427  0.0  1.0    373.0  bm25
2             Question Answering  0.458546  0.349367  0.0  1.0  12335.0  bm25
3   Duplicate Question Retrieval  0.769127  0.327340  0.0  1.0  10000.0  bm25
4               Entity Retrieval  0.309808  0.269749  0.0  1.0    400.0  bm25
5            Citation-Prediction  0.144143  0.203627  0.0  1.0   1000.0  bm25
6                  Fact Checking  0.523770  0.402584  0.0  1.0   8501.0  bm25
7             Argument Retrieval  0.388526  0.348664  0.0  1.0   1455.0  bm25
8   StackExchange Post Retrieval  0.174144  0.272762  0.0  1.0    749.0  bm25
9                 Code Retrieval  0.169408  0.275190  0.0  1.0    254.0  bm25
10             Theorem Re

In [ ]:
# Performance statistics based on query type
 

query_type_list = ["FACTOID", "INSTRUCTION", "REASON", "EVIDENCE-BASED", "COMPARISON", "EXPERIENCE", "DEBATE",
                   "NOT-A-QUESTION"]

for model in model_name_list:
    col = f"{model}_score_ndcg@10"
    grouped_query_type = data.groupby(['query_type'])[col].agg(
        ['mean', 'std', 'min', 'max', 'count']).reset_index()
    grouped_query_type['model'] = model

        # Calculate the average performance of query_type-level
    overall_mean = grouped_query_type['mean'].mean()
    print(f"Query Type-Level Overall Mean for model {model}: {overall_mean}")
    #Add a row to calculate statistics for all queries.
    overall_stats = data[col].agg(['mean', 'std', 'min', 'max', 'count']).to_frame().T
    overall_stats['query_type'] = 'ALL-query-level'
    overall_stats['model'] = model
    grouped_query_type = pd.concat([grouped_query_type, overall_stats], ignore_index=True)

    # Reorder the rows according to the query_type_list, while retaining the newly added 'ALL' rows.
    grouped_query_type = grouped_query_type.set_index('query_type').reindex(query_type_list + ['ALL-query-level']).reset_index()

    print(f"Model: {model} - Query Type Performance:")
    print(grouped_query_type)
    print("-" * 50)


Query Type-Level Overall Mean for model bm25: 0.5819432443997767
Model: bm25 - Query Type Performance:
        query_type      mean       std  min  max    count model
0          FACTOID  0.425245  0.362206  0.0  1.0  15887.0  bm25
1      INSTRUCTION  0.627936  0.397576  0.0  1.0   2708.0  bm25
2           REASON  0.639736  0.403266  0.0  1.0   1381.0  bm25
3   EVIDENCE-BASED  0.397188  0.417291  0.0  1.0   5589.0  bm25
4       COMPARISON  0.703788  0.346235  0.0  1.0    539.0  bm25
5       EXPERIENCE  0.737081  0.345452  0.0  1.0   1582.0  bm25
6           DEBATE  0.672417  0.364307  0.0  1.0   3257.0  bm25
7   NOT-A-QUESTION  0.452155  0.402790  0.0  1.0  11581.0  bm25
8  ALL-query-level  0.482823  0.397378  0.0  1.0  42524.0  bm25
--------------------------------------------------
Query Type-Level Overall Mean for model contriever: 0.6899849921767358
Model: contriever - Query Type Performance:
        query_type      mean       std  min  max    count       model
0          FACTOID  0

In [ ]:
# Separate the datasets by corpus source type and compute the performance statistics for each corpus category
corpus_types = {
    "msmarco":"General Web",
    "trec-covid":"Scientific Paper",
    "nfcorpus": "Scientific Paper",
    "nq": "Wikipedia",
    "hotpotqa":"Wikipedia",
    "fiqa":"Online Community",
    "quora":"Online Community",
    "dbpedia-entity":"Wikipedia",
    "scidocs": "Scientific Paper",
    "scifact": "Scientific Paper",
    "fever":"Wikipedia",
    "climate-fever": "Wikipedia",
    "arguana":"Domain KB",
    "webis-touche2020":"Domain KB",
    "biology":"Online Community",
    "earth_science":"Online Community",
    "economics":"Online Community",
    "psychology" :"Online Community",
    "robotics":"Online Community",
    "stackoverflow":"Online Community",
    "sustainable_living":"Online Community",
    "leetcode":'Domain KB',
    "pony":"Domain KB",
    "aops" :"Domain KB",
    "theoremqa_theorems" :"Domain KB",
    "theoremqa_questions" :"Domain KB",
    "browsecomp_plus":"General Web",
}

corpus_types_list = ["General Web", "Scientific Paper", "Wikipedia", "Online Community",  "Domain KB", ]

# Add a corpus_type column
data['corpus_type'] = data['dataset_name'].map(corpus_types)
data


,query_id,dataset_name,raw_query_id,query_type,query_text,split,bm25_score_ndcg@10,contriever_score_ndcg@10,bge_m3_score_ndcg@10,qwen3_score_ndcg@10,linq_score_ndcg@10,gte_score_ndcg@10,reasonir_score_ndcg@10,diver_score_ndcg@10,bge_reasoner_score_ndcg@10,task_type,corpus_type
0,trec-covid__test__1,trec-covid,1,EVIDENCE-BASED,what is the origin of COVID-19,test,0.252841,0.039199,0.340113,1.000000,0.716937,0.784617,0.653408,0.501819,0.778429,Bio-Medical Retrieval,Scientific Paper
1,trec-covid__test__2,trec-covid,2,EVIDENCE-BASED,how does the coronavirus respond to changes in...,test,0.613283,0.252841,0.864315,1.000000,1.000000,0.861138,0.926636,1.000000,0.921602,Bio-Medical Retrieval,Scientific Paper
2,trec-covid__test__3,trec-covid,3,DEBATE,will SARS-CoV2 infected people develop immunit...,test,0.195189,0.894275,0.000000,1.000000,0.930569,0.899030,0.827749,0.826422,0.650414,Bio-Medical Retrieval,Scientific Paper
3,trec-covid__test__4,trec-covid,4,EVIDENCE-BASED,what causes death from Covid-19?,test,0.135685,0.110046,0.314880,1.000000,0.966873,0.965284,0.591945,0.957428,0.831848,Bio-Medical Retrieval,Scientific Paper
4,trec-covid__test__5,trec-covid,5,EVIDENCE-BASED,what drugs have been active against SARS-CoV o...,test,0.519520,0.508287,0.283713,1.000000,0.743345,0.546287,0.706663,0.671759,0.571936,Bio-Medical Retrieval,Scientific Paper
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42519,msmarco__test__1132532,msmarco,1132532,FACTOID,average annual income data analyst,test,0.428797,0.835746,0.889954,1.000000,0.978793,0.976856,0.776182,0.770499,0.898255,Passage Retrieval,General Web
42520,msmarco__test__1133579,msmarco,1133579,EVIDENCE-BASED,how does granulation tissue start,test,0.554417,0.629336,0.812214,0.818852,0.817511,0.627877,0.652478,0.676379,0.741429,Passage Retrieval,General Web
42521,msmarco__test__1136043,msmarco,1136043,FACTOID,difference between a hotel and motel,test,0.346317,0.805447,0.724774,0.725397,0.724774,0.723278,0.618023,0.683002,0.593403,Passage Retrieval,General Web
42522,msmarco__test__1136047,msmarco,1136047,FACTOID,difference between a company's strategy and bu...,test,0.459095,0.327582,0.475790,0.557774,0.613942,0.371733,0.433620,0.461970,0.305609,Passage Retrieval,General Web


In [ ]:
for model in model_name_list:
    col = f"{model}_score_ndcg@10"
    grouped_corpus_type = data.groupby(['corpus_type'])[col].agg(
        ['mean', 'std', 'min', 'max', 'count']).reset_index()
    grouped_corpus_type['model'] = model

        # # Calculate the average performance of corpus_type-level
    overall_mean = grouped_corpus_type['mean'].mean()
    print(f"Task-Level Overall Mean for model {model}: {overall_mean}")
 
    overall_stats = data[col].agg(['mean', 'std', 'min', 'max', 'count']).to_frame().T
    overall_stats['corpus_type'] = 'ALL-query-level'
    overall_stats['model'] = model
    grouped_corpus_type = pd.concat([grouped_corpus_type, overall_stats], ignore_index=True)

    # Reorder the rows according to the order of courpus_type_list, while retaining the newly added 'ALL' rows.
    grouped_corpus_type = grouped_corpus_type.set_index('corpus_type').reindex(corpus_types_list + ['ALL-query-level']).reset_index()

    print(f"Model: {model} - corpus_type Type Performance:")
    print(grouped_corpus_type)
    print("-" * 50)

Task-Level Overall Mean for model bm25: 0.3993991529807195
Model: bm25 - corpus_type Type Performance:
        corpus_type      mean       std  min  max    count model
0       General Web  0.202677  0.315625  0.0  1.0   7907.0  bm25
1  Scientific Paper  0.282876  0.335903  0.0  1.0   1673.0  bm25
2         Wikipedia  0.505786  0.365872  0.0  1.0  19458.0  bm25
3  Online Community  0.699755  0.373187  0.0  1.0  11397.0  bm25
4         Domain KB  0.305902  0.344247  0.0  1.0   2089.0  bm25
5   ALL-query-level  0.482823  0.397378  0.0  1.0  42524.0  bm25
--------------------------------------------------
Task-Level Overall Mean for model contriever: 0.4847872058944619
Model: contriever - corpus_type Type Performance:
        corpus_type      mean       std  min  max    count       model
0       General Web  0.374858  0.374119  0.0  1.0   7907.0  contriever
1  Scientific Paper  0.301566  0.338314  0.0  1.0   1673.0  contriever
2         Wikipedia  0.617755  0.341853  0.0  1.0  19458.0  con

In [9]:
data

,query_id,dataset_name,raw_query_id,query_type,query_text,split,bm25_score_ndcg@10,contriever_score_ndcg@10,bge_m3_score_ndcg@10,qwen3_score_ndcg@10,linq_score_ndcg@10,gte_score_ndcg@10,reasonir_score_ndcg@10,diver_score_ndcg@10,bge_reasoner_score_ndcg@10,task_type,corpus_type
0,trec-covid__test__1,trec-covid,1,EVIDENCE-BASED,what is the origin of COVID-19,test,0.252841,0.039199,0.340113,1.000000,0.716937,0.784617,0.653408,0.501819,0.778429,Bio-Medical Retrieval,Scientific Paper
1,trec-covid__test__2,trec-covid,2,EVIDENCE-BASED,how does the coronavirus respond to changes in...,test,0.613283,0.252841,0.864315,1.000000,1.000000,0.861138,0.926636,1.000000,0.921602,Bio-Medical Retrieval,Scientific Paper
2,trec-covid__test__3,trec-covid,3,DEBATE,will SARS-CoV2 infected people develop immunit...,test,0.195189,0.894275,0.000000,1.000000,0.930569,0.899030,0.827749,0.826422,0.650414,Bio-Medical Retrieval,Scientific Paper
3,trec-covid__test__4,trec-covid,4,EVIDENCE-BASED,what causes death from Covid-19?,test,0.135685,0.110046,0.314880,1.000000,0.966873,0.965284,0.591945,0.957428,0.831848,Bio-Medical Retrieval,Scientific Paper
4,trec-covid__test__5,trec-covid,5,EVIDENCE-BASED,what drugs have been active against SARS-CoV o...,test,0.519520,0.508287,0.283713,1.000000,0.743345,0.546287,0.706663,0.671759,0.571936,Bio-Medical Retrieval,Scientific Paper
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42519,msmarco__test__1132532,msmarco,1132532,FACTOID,average annual income data analyst,test,0.428797,0.835746,0.889954,1.000000,0.978793,0.976856,0.776182,0.770499,0.898255,Passage Retrieval,General Web
42520,msmarco__test__1133579,msmarco,1133579,EVIDENCE-BASED,how does granulation tissue start,test,0.554417,0.629336,0.812214,0.818852,0.817511,0.627877,0.652478,0.676379,0.741429,Passage Retrieval,General Web
42521,msmarco__test__1136043,msmarco,1136043,FACTOID,difference between a hotel and motel,test,0.346317,0.805447,0.724774,0.725397,0.724774,0.723278,0.618023,0.683002,0.593403,Passage Retrieval,General Web
42522,msmarco__test__1136047,msmarco,1136047,FACTOID,difference between a company's strategy and bu...,test,0.459095,0.327582,0.475790,0.557774,0.613942,0.371733,0.433620,0.461970,0.305609,Passage Retrieval,General Web


In [ ]:
 

data.columns


Index(['query_id', 'dataset_name', 'raw_query_id', 'query_type', 'query_text',
       'split', 'bm25_score_ndcg@10', 'contriever_score_ndcg@10',
       'bge_m3_score_ndcg@10', 'qwen3_score_ndcg@10', 'linq_score_ndcg@10',
       'gte_score_ndcg@10', 'reasonir_score_ndcg@10', 'diver_score_ndcg@10',
       'bge_reasoner_score_ndcg@10', 'task_type', 'corpus_type'],
      dtype='object')

In [ ]:
import pandas as pd
import pingouin as pg
import statsmodels.formula.api as smf

 

# 1. Define your 8 model columns
model_columns = ['bm25_score_ndcg@10',
    'contriever_score_ndcg@10', 'bge_m3_score_ndcg@10', 
    'qwen3_score_ndcg@10', 'linq_score_ndcg@10', 
    'gte_score_ndcg@10', 'reasonir_score_ndcg@10', 
    'diver_score_ndcg@10', 'bge_reasoner_score_ndcg@10'
]

# 2. Define the ID and factor columns required for the analysis.
id_vars = [
    'query_id', 'dataset_name', 'query_type', 
    'task_type', 'corpus_type'
]

# 3. Use pd.melt() to convert wide data to long data.
df_long = pd.melt(
    data,
    id_vars=id_vars,
    value_vars=model_columns,
    var_name='Model_Raw',  # Temporary column name containing '..._score_ndcg@10'
    value_name='ndcg@10'     # Your dependent variable (Y)
)

# 4.Clean up model name
# (For example, 'contriever_score_ndcg@10' -> 'contriever')
df_long['Model'] = df_long['Model_Raw'].str.replace('_score_ndcg@10', '')

# 5. Discard temporary original model columns
df_long = df_long.drop(columns=['Model_Raw'])

# 6. Handle missing values ​​(if any).
df_long = df_long.dropna(subset=['ndcg@10'])

print("Long format data is ready (df_long):")
print(df_long.head())

# Saving Data

# 3. Saving the DataFrame as a CSV File

# Setting index=False is a good practice to prevent pandas from saving the DataFrame's indices (0, 1, 2...) as an extra column in the CSV file.

df_long.to_csv("data/df_long_all_model_scores.csv", index=False)
print(f"\n✅ data saved: data/df_long_all_model_scores.csv")


长格式数据准备完毕 (df_long):
              query_id dataset_name      query_type              task_type  \
0  trec-covid__test__1   trec-covid  EVIDENCE-BASED  Bio-Medical Retrieval   
1  trec-covid__test__2   trec-covid  EVIDENCE-BASED  Bio-Medical Retrieval   
2  trec-covid__test__3   trec-covid          DEBATE  Bio-Medical Retrieval   
3  trec-covid__test__4   trec-covid  EVIDENCE-BASED  Bio-Medical Retrieval   
4  trec-covid__test__5   trec-covid  EVIDENCE-BASED  Bio-Medical Retrieval   

        corpus_type   ndcg@10 Model  
0  Scientific Paper  0.252841  bm25  
1  Scientific Paper  0.613283  bm25  
2  Scientific Paper  0.195189  bm25  
3  Scientific Paper  0.135685  bm25  
4  Scientific Paper  0.519520  bm25  

✅ 数据已成功保存到: data/df_long_all_model_scores.csv


In [12]:
df_long.shape

(382716, 7)